# Exercícios - 13/03/2026
Detecção de Outliers com IQR

## Exercício 1
Percentis 25, 50 e 75 do vetor de tempo de processamento

In [1]:
import numpy as np

tempos = [12, 15, 14, 13, 16, 12, 14, 150, 13, 15]

q25 = np.percentile(tempos, 25)
q50 = np.percentile(tempos, 50)
q75 = np.percentile(tempos, 75)

print(f'Percentil 25 (Q1): {q25}')
print(f'Percentil 50 (Q2/Mediana): {q50}')
print(f'Percentil 75 (Q3): {q75}')

Percentil 25 (Q1): 13.0
Percentil 50 (Q2/Mediana): 14.0
Percentil 75 (Q3): 15.0


## Exercício 2
IQR, Limite Inferior e Superior (constante 1.5)

In [2]:
iqr = q75 - q25
limite_inferior = q25 - 1.5 * iqr
limite_superior = q75 + 1.5 * iqr

print(f'IQR: {iqr}')
print(f'Limite Inferior: {limite_inferior}')
print(f'Limite Superior: {limite_superior}')

IQR: 2.0
Limite Inferior: 10.0
Limite Superior: 18.0


## Exercício 3
Q1 e Q3 manuais para lista com número par de elementos (sem funções prontas do NumPy)

In [3]:
dados = [100, 150, 200, 250, 300, 350]
n = len(dados)
meio = n // 2

if n % 2 == 0:
    metade_inferior = dados[:meio]
    metade_superior = dados[meio:]
else:
    metade_inferior = dados[:meio]
    metade_superior = dados[meio+1:]

# Mediana de cada metade
def mediana(lista):
    n = len(lista)
    meio = n // 2
    if n % 2 == 0:
        return (lista[meio - 1] + lista[meio]) / 2
    else:
        return lista[meio]

Q1 = mediana(metade_inferior)
Q3 = mediana(metade_superior)

print(f'Metade inferior: {metade_inferior}')
print(f'Metade superior: {metade_superior}')
print(f'Q1: {Q1}')
print(f'Q3: {Q3}')

Metade inferior: [100, 150, 200]
Metade superior: [250, 300, 350]
Q1: 150
Q3: 300


## Exercício 4
Anomalias nas medições de tensão elétrica

In [4]:
tensao = [110, 115, 120, 118, 112, 220, 116, 114, 119, 12]

q1_t = np.percentile(tensao, 25)
q3_t = np.percentile(tensao, 75)
iqr_t = q3_t - q1_t
li_t = q1_t - 1.5 * iqr_t
ls_t = q3_t + 1.5 * iqr_t

print(f'Q1={q1_t}, Q3={q3_t}, IQR={iqr_t}')
print(f'Limite Inferior={li_t}, Limite Superior={ls_t}')

anomalias = [v for v in tensao if v < li_t or v > ls_t]
print(f'Valores anômalos: {anomalias}')

Q1=112.5, Q3=118.75, IQR=6.25
Limite Inferior=103.125, Limite Superior=128.125
Valores anômalos: [220, 12]


## Exercício 5
Função `detectar_anomalias(dados, multiplicador)`

In [5]:
def detectar_anomalias(dados, multiplicador):
    q1 = np.percentile(dados, 25)
    q3 = np.percentile(dados, 75)
    iqr = q3 - q1
    limite_inferior = q1 - multiplicador * iqr
    limite_superior = q3 + multiplicador * iqr
    return [v for v in dados if v < limite_inferior or v > limite_superior]

## Exercício 6
Testando a função com novo vetor

In [6]:
vetor = [45, 50, 55, 60, 48, 52, 51, 98, 49, 53]
outliers = detectar_anomalias(vetor, 1.5)
print(f'Outliers encontrados: {outliers}')

Outliers encontrados: [98]


## Exercício 7
DataFrame de uso de memória com Pandas e cálculo do IQR via `.quantile()`

In [7]:
import pandas as pd

df_maquinas = pd.DataFrame({
    'ID_Maquina': [1, 2, 3, 4, 5],
    'Uso_Memoria_MB': [2048, 2100, 2050, 8192, 2080]
})

q1_mem = df_maquinas['Uso_Memoria_MB'].quantile(0.25)
q3_mem = df_maquinas['Uso_Memoria_MB'].quantile(0.75)
iqr_mem = q3_mem - q1_mem

print(df_maquinas)
print(f'\nQ1={q1_mem}, Q3={q3_mem}, IQR={iqr_mem}')

   ID_Maquina  Uso_Memoria_MB
0           1            2048
1           2            2100
2           3            2050
3           4            8192
4           5            2080

Q1=2050.0, Q3=2100.0, IQR=50.0


## Exercício 8
Máscara booleana para filtrar linhas normais (sem anomalias)

In [8]:
li_mem = q1_mem - 1.5 * iqr_mem
ls_mem = q3_mem + 1.5 * iqr_mem

mascara = (df_maquinas['Uso_Memoria_MB'] >= li_mem) & (df_maquinas['Uso_Memoria_MB'] <= ls_mem)
df_normal = df_maquinas[mascara]

print(f'Limite Inferior={li_mem}, Limite Superior={ls_mem}')
print('\nMáquinas dentro da normalidade:')
print(df_normal)

Limite Inferior=1975.0, Limite Superior=2175.0

Máquinas dentro da normalidade:
   ID_Maquina  Uso_Memoria_MB
0           1            2048
1           2            2100
2           3            2050
4           5            2080


## Exercício 9
Substituir outlier (300) pela mediana usando `np.where()`

In [9]:
df_temp = pd.DataFrame({'temperatura': [80, 82, 85, 81, 300, 83]})

q1_tmp = np.percentile(df_temp['temperatura'], 25)
q3_tmp = np.percentile(df_temp['temperatura'], 75)
iqr_tmp = q3_tmp - q1_tmp
li_tmp = q1_tmp - 1.5 * iqr_tmp
ls_tmp = q3_tmp + 1.5 * iqr_tmp

mediana_temp = np.percentile(df_temp['temperatura'], 50)

df_temp['temperatura'] = np.where(
    (df_temp['temperatura'] < li_tmp) | (df_temp['temperatura'] > ls_tmp),
    mediana_temp,
    df_temp['temperatura']
)

print(f'Mediana usada para substituição: {mediana_temp}')
print(df_temp)

Mediana usada para substituição: 82.5
   temperatura
0         80.0
1         82.0
2         85.0
3         81.0
4         82.5
5         83.0


## Exercício 10
IQR por grupo com `.groupby()` e detecção de anomalias por sensor

In [10]:
df_sensores = pd.DataFrame({
    'Sensor_ID': ['A','A','A','A','A','B','B','B','B','B'],
    'Valor_Leitura': [10, 12, 11, 13, 100, 200, 205, 198, 202, 500]
})

def detectar_por_grupo(grupo):
    q1 = grupo['Valor_Leitura'].quantile(0.25)
    q3 = grupo['Valor_Leitura'].quantile(0.75)
    iqr = q3 - q1
    li = q1 - 1.5 * iqr
    ls = q3 + 1.5 * iqr
    anomalias = grupo[(grupo['Valor_Leitura'] < li) | (grupo['Valor_Leitura'] > ls)]
    return anomalias

resultado = df_sensores.groupby('Sensor_ID', group_keys=False).apply(detectar_por_grupo)

print('Anomalias por sensor:')
print(resultado)

Anomalias por sensor:
   Valor_Leitura
4            100
9            500


## Exercício 11
Carregar `dados_sensores.csv`, verificar NaN e preencher com mediana

In [11]:
df_csv = pd.read_csv('dados_sensores.csv')

print('Valores NaN por coluna:')
print(df_csv.isna().sum())

df_csv['temperatura_celsius'] = df_csv['temperatura_celsius'].fillna(df_csv['temperatura_celsius'].median())
df_csv['pressao_psi'] = df_csv['pressao_psi'].fillna(df_csv['pressao_psi'].median())

print('\nApós preenchimento com mediana:')
print(df_csv.isna().sum())
print(df_csv.head())

Valores NaN por coluna:
id_leitura             0
sensor_id              0
temperatura_celsius    2
pressao_psi            2
dtype: int64

Após preenchimento com mediana:
id_leitura             0
sensor_id              0
temperatura_celsius    0
pressao_psi            0
dtype: int64
   id_leitura sensor_id  temperatura_celsius  pressao_psi
0           1      S-03            36.164246    98.377545
1           2      S-01            36.775497    98.970666
2           3      S-03            36.788665    92.799793
3           4      S-03            36.509996   105.953636
4           5      S-01            34.585668   106.496984


## Exercício 12
Remover outliers por IQR nas colunas de temperatura e pressão, exportar para `dados_validados.csv`

In [12]:
def limites_iqr(coluna):
    q1 = coluna.quantile(0.25)
    q3 = coluna.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

li_temp, ls_temp = limites_iqr(df_csv['temperatura_celsius'])
li_pres, ls_pres = limites_iqr(df_csv['pressao_psi'])

print(f'Temperatura  -> LI: {li_temp:.2f}, LS: {ls_temp:.2f}')
print(f'Pressão      -> LI: {li_pres:.2f}, LS: {ls_pres:.2f}')

mascara_temp = (df_csv['temperatura_celsius'] >= li_temp) & (df_csv['temperatura_celsius'] <= ls_temp)
mascara_pres = (df_csv['pressao_psi'] >= li_pres) & (df_csv['pressao_psi'] <= ls_pres)

df_validado = df_csv[mascara_temp & mascara_pres]

print(f'\nLinhas antes: {len(df_csv)} | Linhas após remoção de outliers: {len(df_validado)}')

df_validado.to_csv('dados_validados.csv', index=False)
print('Arquivo dados_validados.csv exportado com sucesso!')

Temperatura  -> LI: 30.49, LS: 40.13
Pressão      -> LI: 87.37, LS: 114.89

Linhas antes: 100 | Linhas após remoção de outliers: 95
Arquivo dados_validados.csv exportado com sucesso!
